In [12]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

import plotly.express as px


In [2]:
query_location = './sql/organic_installs.sql'
parameters = {
    'start_date':'2026-01-01',
}


bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.13 GB when run.
Estimated query cost: $0.01


In [14]:
refresh_data = False
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query='./sql/organic_installs.sql', is_path=True, query_parameters=parameters)
else:
    data = pd.read_pickle('./data/organic_installs_data.pkl')

In [15]:
data

,user_id,install_dt,platform,display_campaign_network
0,3F3B27DAFEA98BB4,2026-04-30,IOS,FACEBOOK
1,E64A8FC63CD5AA1A,2026-04-22,IOS,Non-Attributed
2,EA5C006C6CC4DE1A,2026-05-13,IOS,Non-Attributed
3,D2429BEFC0FBC5EC,2026-04-29,IOS,EXMOX
4,46208190475D5D47,2026-05-19,IOS,EXMOX
...,...,...,...,...
277347,3B820EDAB4AA14E7,2026-05-14,IOS,EXMOX
277348,16988D5E7FC11A3F,2026-05-14,IOS,EXMOX
277349,6715A1A92792616D,2026-05-14,IOS,Non-Attributed
277350,B1AC42FF921069A1,2026-05-14,IOS,FACEBOOK


In [16]:
attribution = pd.DataFrame()
attribution = data.copy()
attribution['attribution_type'] = ['Non-Attributed' if x == 'Non-Attributed' else 'Attributed' for x in attribution['display_campaign_network']]
attribution

,user_id,install_dt,platform,display_campaign_network,attribution_type
0,3F3B27DAFEA98BB4,2026-04-30,IOS,FACEBOOK,Attributed
1,E64A8FC63CD5AA1A,2026-04-22,IOS,Non-Attributed,Non-Attributed
2,EA5C006C6CC4DE1A,2026-05-13,IOS,Non-Attributed,Non-Attributed
3,D2429BEFC0FBC5EC,2026-04-29,IOS,EXMOX,Attributed
4,46208190475D5D47,2026-05-19,IOS,EXMOX,Attributed
...,...,...,...,...,...
277347,3B820EDAB4AA14E7,2026-05-14,IOS,EXMOX,Attributed
277348,16988D5E7FC11A3F,2026-05-14,IOS,EXMOX,Attributed
277349,6715A1A92792616D,2026-05-14,IOS,Non-Attributed,Non-Attributed
277350,B1AC42FF921069A1,2026-05-14,IOS,FACEBOOK,Attributed


In [21]:
attribution.display_campaign_network.unique()

array(['FACEBOOK', 'Non-Attributed', 'EXMOX', 'LIFTOFF', 'ADJOE', 'pixly',
       'Ironsource', 'Unity', 'InfluenceMobile', 'GOOGLE', 'MINTEGRAL',
       'MISTPLAY', 'Youappi', 'TIKTOK', 'Adikteev', 'MOLOCO', 'Applovin',
       'Vungle', 'DigitalTurbine', 'Social_Instagram', 'Copper',
       'mediabodies', 'AppleSearch', 'Benjamin', 'cherrypick', 'Tapjoy',
       'Social_ProjectFuture', 'Social_Tiktok', 'Prodege', 'brainlabs',
       'Social_Facebook', 'Remerge', 'Podsights', 'Social_Youtube',
       'goat'], dtype=object)

In [22]:
attribution_users = attribution.groupby(['install_dt', 'attribution_type']).agg(
    unique_users = ('user_id', 'nunique')
).reset_index()

attribution_users_pf = attribution[attribution['display_campaign_network'] == 'Social_ProjectFuture'].groupby(['install_dt', 'attribution_type']).agg(
    unique_users = ('user_id', 'nunique')
).reset_index()

attribution_users

,install_dt,attribution_type,unique_users
0,2026-01-01,Attributed,1327
1,2026-01-01,Non-Attributed,1268
2,2026-01-02,Attributed,1427
3,2026-01-02,Non-Attributed,1439
4,2026-01-03,Attributed,1334
...,...,...,...
275,2026-05-18,Non-Attributed,671
276,2026-05-19,Attributed,587
277,2026-05-19,Non-Attributed,713
278,2026-05-20,Attributed,20


In [20]:
# hide-output
fig = px.line(attribution_users, 
              x='install_dt', 
              y='unique_users',
              color='attribution_type',
              title='Daily Unique Users by Event',
              width=1200,
              height=600,
              hover_data={'unique_users': True})

#fig = add_vlines_to_figure(fig, vlines_events)

#fig.update_yaxes(rangemode='tozero')
fig.show()

In [23]:
# hide-output
fig = px.line(attribution_users_pf, 
              x='install_dt', 
              y='unique_users',
              color='attribution_type',
              title='Daily Unique Users by Event',
              width=1200,
              height=600,
              hover_data={'unique_users': True})

#fig = add_vlines_to_figure(fig, vlines_events)

#fig.update_yaxes(rangemode='tozero')
fig.show()